#  Kaggle HBAAC 2026 — Demand Forecasting (v5)

An optimized ensemble solution blending a regression LightGBM model (trained on square-root targets) and a Tweedie LightGBM model. Features advanced cross-feature interaction terms, optimized lag/EWM aggregates, trend indicators, holiday proximity encodings, and a data-driven post-processing calibration layer based on validation residuals.

---
##  Cell 1 — Libraries & Constants

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

INPUT_DIR  = Path('/kaggle/input/competitions/hbaac-round2')
OUTPUT_DIR = Path('/kaggle/working')
TRAIN_PATH  = INPUT_DIR / 'train.csv'
SAMPLE_PATH = INPUT_DIR / 'sample_submission.csv'

TRAIN_START = pd.Timestamp('2020-11-17')
TRAIN_END   = pd.Timestamp('2025-09-05')
VAL_START   = pd.Timestamp('2025-09-06')
VAL_END     = pd.Timestamp('2025-10-03')
EVAL_START  = pd.Timestamp('2025-10-04')
EVAL_END    = pd.Timestamp('2025-10-31')
SPLIT_DATE  = pd.Timestamp('2025-07-12')
HORIZON     = 56
SEED        = 42
print('Libraries loaded')

---
## ️ Cell 2 — Load & Parse Raw Data

In [ ]:
def parse_vnd(series: pd.Series) -> pd.Series:
    """Parse Vietnamese decimal-comma number strings to float."""
    return (
        series.astype(str).str.strip()
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .replace({'nan': np.nan, '': np.nan})
        .astype(float)
    )

print('Reading train.csv ...')
raw = pd.read_csv(
    TRAIN_PATH,
    dtype={'Stt': 'str', 'ItemCode': 'str', 'UnitPrice': 'str', 'Unit Cost': 'str'},
    parse_dates=['Date'],
    low_memory=False
)
raw['Quantity']    = pd.to_numeric(raw['Quantity'],    errors='coerce').fillna(0).astype('int32')
raw['SalesAmount'] = pd.to_numeric(raw['SalesAmount'], errors='coerce').fillna(0).astype('int64')
raw['Cost Amount'] = pd.to_numeric(raw['Cost Amount'], errors='coerce').fillna(0).astype('int64')
raw['UnitPrice'] = parse_vnd(raw['UnitPrice'])
raw['Unit Cost'] = parse_vnd(raw['Unit Cost'])
raw['ItemCode'] = raw['ItemCode'].astype('category')

print(f'Raw rows : {len(raw):,}')
print(f'SKUs     : {raw["ItemCode"].nunique():,}')
print(f'Dates    : {raw["Date"].min().date()} to {raw["Date"].max().date()}')

---
##  Cell 3 — Per-SKU Profit Weights

In [ ]:
raw['Profit'] = raw['SalesAmount'].astype(float) - raw['Cost Amount'].astype(float)
sku_profit = (
    raw.groupby('ItemCode', observed=True)['Profit']
       .sum().reset_index().rename(columns={'Profit': 'total_profit'})
)
sku_profit['total_profit'] = sku_profit['total_profit'].clip(lower=0)
total = sku_profit['total_profit'].sum()
sku_profit['weight'] = sku_profit['total_profit'] / total if total > 0 else 0.0
print(f'SKUs with positive profit : {(sku_profit["total_profit"] > 0).sum():,}')
print(f'Total profit (VND)        : {total:,.0f}')
sku_profit.nlargest(5, 'weight')[['ItemCode', 'total_profit', 'weight']]

---
##  Cell 4 — Handle Returns & Aggregate Daily

In [ ]:
daily = (
    raw.groupby(['ItemCode', 'Date'], observed=True)
    .agg(
        Quantity=('Quantity', 'sum'),
        SalesAmount=('SalesAmount', 'sum'),
        CostAmount=('Cost Amount', 'sum'),
    )
    .reset_index()
)
daily['Quantity'] = daily['Quantity'].clip(lower=0)
print(f'Daily grain rows: {len(daily):,}')

---
##  Cell 5 — Reindex to Full Daily Calendar

In [ ]:
all_dates = pd.date_range(TRAIN_START, TRAIN_END, freq='D')
all_skus  = daily['ItemCode'].cat.categories if hasattr(daily['ItemCode'], 'cat') \
            else daily['ItemCode'].unique()
full_idx = pd.MultiIndex.from_product([all_skus, all_dates], names=['ItemCode', 'Date'])
print(f'Full grid: {len(full_idx):,} rows')

daily = (
    daily.set_index(['ItemCode', 'Date'])
    .reindex(full_idx, fill_value=0)
    .reset_index()
)
daily['ItemCode'] = daily['ItemCode'].astype('category')
daily['Quantity'] = daily['Quantity'].astype('float32')
print(f'After reindex: {len(daily):,} rows')

---
## ️ Cell 6 — Vietnamese Holidays

In [ ]:
def get_vietnamese_holidays(start_year=2020, end_year=2026):
    holidays = []
    fixed = ['01-01', '04-30', '05-01', '09-02']
    lunar_new_year_day1 = {
        2020: '2020-01-25', 2021: '2021-02-12', 2022: '2022-02-01',
        2023: '2023-01-22', 2024: '2024-02-10', 2025: '2025-01-29',
        2026: '2026-02-17',
    }
    hung_kings = {
        2020: '2020-04-02', 2021: '2021-04-21', 2022: '2022-04-10',
        2023: '2023-04-29', 2024: '2024-04-18', 2025: '2025-04-07',
        2026: '2026-04-26',
    }
    tet_day1_dates = {}
    for year in range(start_year, end_year + 1):
        for mmdd in fixed:
            holidays.append(pd.Timestamp(f'{year}-{mmdd}'))
        if year in lunar_new_year_day1:
            d1 = pd.Timestamp(lunar_new_year_day1[year])
            tet_day1_dates[year] = d1
            shift_days = (d1.dayofweek + 2) % 7
            start_date = d1 - pd.Timedelta(days=shift_days)
            end_date   = start_date + pd.Timedelta(days=8)
            holidays.extend(pd.date_range(start_date, end_date, freq='D').tolist())
        if year in hung_kings:
            holidays.append(pd.Timestamp(hung_kings[year]))
    return pd.DatetimeIndex(sorted(set(holidays))), tet_day1_dates

VN_HOLIDAYS, TET_DAY1_DICT = get_vietnamese_holidays(2020, 2026)
TET_DAY1_LIST = sorted(TET_DAY1_DICT.values())
print(f'Holiday dates: {len(VN_HOLIDAYS)}')

---
## ️ Cell 7 — Feature Engineering (v5: Targeted on FI Analysis)

In [ ]:
def build_features(df, vn_holidays, tet_day1_list):
    """Build optimized feature set for demand forecasting, leveraging lag features, multi-span EWM, trend signals, rolling window aggregates, holiday/Tet proximity, and pricing/discount flags."""
    df = df.sort_values(['ItemCode', 'Date']).copy()

    # ── 1. Core Time Features ────────────────────────────────────────────────
    df['dayofweek']   = df['Date'].dt.dayofweek.astype('int8')
    df['day']         = df['Date'].dt.day.astype('int8')
    df['month']       = df['Date'].dt.month.astype('int8')
    df['year']        = df['Date'].dt.year.astype('int16')
    df['quarter']     = df['Date'].dt.quarter.astype('int8')
    df['weekofyear']  = df['Date'].dt.isocalendar().week.astype('int8')
    df['is_weekend']  = (df['dayofweek'] >= 5).astype('int8')
    df['is_holiday']  = df['Date'].isin(vn_holidays).astype('int8')

    # Continuous trend feature (days from TRAIN_START)
    TRAIN_START_TS = df['Date'].min()
    df['trend_t'] = (df['Date'] - TRAIN_START_TS).dt.days.astype('int16')

    # ── 2. Holiday Proximity (keep key ones from v4, prune weak ones) ────────
    holiday_dates_sorted = sorted(vn_holidays)
    unique_dates = df['Date'].unique()

    next_hol_map  = {}
    since_hol_map = {}
    pre7_map      = {}
    post3_map     = {}
    days_to_tet_map    = {}
    days_since_tet_map = {}

    for d in unique_dates:
        future_hols  = [h for h in holiday_dates_sorted if h > d]
        past_hols    = [h for h in holiday_dates_sorted if h < d]
        d2n  = (future_hols[0] - d).days if future_hols else 365
        d4l  = (d - past_hols[-1]).days  if past_hols  else 365

        next_hol_map[d]  = d2n
        since_hol_map[d] = d4l
        pre7_map[d]   = 1 if (0 < d2n  <= 7) else 0
        post3_map[d]  = 1 if (0 < d4l  <= 3) else 0

        future_tet = [t for t in tet_day1_list if t > d]
        past_tet   = [t for t in tet_day1_list if t <= d]
        d_to_t  = (future_tet[0] - d).days if future_tet else 365
        d_from_t= (d - past_tet[-1]).days   if past_tet   else 365
        days_to_tet_map[d]    = min(d_to_t, 365)
        days_since_tet_map[d] = min(d_from_t, 365)

    df['days_to_next_holiday']    = df['Date'].map(next_hol_map).astype('int16')
    df['days_since_last_holiday'] = df['Date'].map(since_hol_map).astype('int16')
    df['is_pre_holiday_7d']       = df['Date'].map(pre7_map).astype('int8')
    df['is_post_holiday_3d']      = df['Date'].map(post3_map).astype('int8')
    df['days_to_tet']             = df['Date'].map(days_to_tet_map).astype('int16')
    df['days_since_tet']          = df['Date'].map(days_since_tet_map).astype('int16')

    # ── 3. Lag Features (pruned to key anchors only) ─────────────────────────
    grp = df.groupby('ItemCode', observed=True)['Quantity']

    # Keep: weekly/biweekly/quarterly/annual anchors (proven useful in v4)
    # Drop: lag_70, lag_84 (weak), lag_357, lag_371 (redundant with lag_364)
    for lag in [56, 63, 91, 364]:
        df[f'lag_{lag}'] = grp.shift(lag).astype('float32')

    # ── 4. EWM Features (expanded — v4 showed these are most valuable) ───────
    # All computed from lag-56 shifted series (no data leakage for 56-day horizon)
    lag56 = grp.shift(56)

    # EWM at multiple spans: short (recent), medium, long (trend)
    for span in [7, 14, 28, 56, 91]:
        df[f'ewm_mean_{span}'] = lag56.transform(
            lambda x, s=span: x.ewm(span=s, min_periods=1).mean()
        ).astype('float32')

    # Ratio features: captures relative position of recent vs trend
    # ewm_ratio_short_long: >1 means accelerating, <1 means decelerating
    eps_r = 1e-3
    df['ewm_ratio_14_56']  = (df['ewm_mean_14']  / (df['ewm_mean_56']  + eps_r)).clip(0, 10).astype('float32')
    df['ewm_ratio_28_91']  = (df['ewm_mean_28']  / (df['ewm_mean_91']  + eps_r)).clip(0, 10).astype('float32')
    df['lag56_vs_ewm56']   = (df['lag_56']        / (df['ewm_mean_56']  + eps_r)).clip(0, 10).astype('float32')

    # ── 5. Rolling Features (keep long window, add burst/frequency) ──────────
    # Keep roll_mean_56 (rank 4 in v4), add max and nonzero-count
    # Drop roll_mean_7, roll_mean_14 (not in top 25), all roll_std (weak)
    for window in [28, 56]:
        df[f'roll_mean_{window}'] = lag56.transform(
            lambda x, w=window: x.rolling(w, min_periods=1).mean()
        ).astype('float32')

    # Roll max: captures burst/peak demand patterns
    df['roll_max_56'] = lag56.transform(
        lambda x: x.rolling(56, min_periods=1).max()
    ).astype('float32')

    # Roll nonzero count: demand frequency in recent window
    df['roll_nonzero_28'] = lag56.transform(
        lambda x: x.rolling(28, min_periods=1).apply(lambda v: (v > 0).sum(), raw=True)
    ).astype('float32')

    # Roll std at key window only (56d — highest gain in v3)
    df['roll_std_56'] = lag56.transform(
        lambda x: x.rolling(56, min_periods=1).std()
    ).fillna(0).astype('float32')

    # ── 6. SKU-Level Statistics ───────────────────────────────────────────────
    n_days = df['Date'].nunique()
    df['sku_sales_freq']  = grp.transform(lambda x: (x > 0).sum() / n_days).astype('float32')
    df['sku_active_days'] = grp.transform(lambda x: (x > 0).sum()).astype('int16')

    # Recent vs overall mean (trend signal)
    # sku_recent_mean: mean of the most recent 90 calendar days for each SKU
    lag_90 = grp.shift(1)
    df['sku_recent_mean_90d'] = lag_90.transform(
        lambda x: x.rolling(90, min_periods=1).mean()
    ).astype('float32')

    # Growth rate: recent 90d mean / all-time mean (captures accelerating SKUs)
    sku_mean_transform = grp.transform('mean')
    df['sku_growth_rate'] = (
        df['sku_recent_mean_90d'] / (sku_mean_transform + 1e-6)
    ).clip(0, 5).astype('float32')

    # Croston streak: days since last sale (from lag_56 perspective)
    def days_since_last_sale_transform(series):
        is_zero = (series == 0).astype(int)
        streak = is_zero.groupby((is_zero != is_zero.shift()).cumsum()).cumsum()
        return streak

    df['lag_56_days_since_sale'] = (
        df.groupby('ItemCode', observed=True)['lag_56']
          .transform(days_since_last_sale_transform)
          .astype('int16')
    )

    # ── 7. Price & Margin ─────────────────────────────────────────────────────
    if 'SalesAmount' in df.columns:
        df['implied_price'] = np.where(df['Quantity'] > 0, df['SalesAmount'] / df['Quantity'], np.nan)
        df['implied_price'] = df.groupby('ItemCode', observed=True)['implied_price'].ffill()
        sku_median_price    = df.groupby('ItemCode', observed=True)['implied_price'].transform('median')
        df['implied_price'] = df['implied_price'].fillna(sku_median_price).fillna(0)

        df['implied_cost'] = np.where(df['Quantity'] > 0, df['CostAmount'] / df['Quantity'], np.nan)
        df['implied_cost'] = df.groupby('ItemCode', observed=True)['implied_cost'].ffill()
        sku_median_cost    = df.groupby('ItemCode', observed=True)['implied_cost'].transform('median')
        df['implied_cost'] = df['implied_cost'].fillna(sku_median_cost).fillna(0)

        df['margin_rate'] = np.where(
            df['implied_price'] > 0,
            (df['implied_price'] - df['implied_cost']) / df['implied_price'], 0.0
        ).astype('float32')
        df['price_norm'] = np.where(
            sku_median_price > 0, df['implied_price'] / sku_median_price, 1.0
        ).astype('float32')

        grp_p = df.groupby('ItemCode', observed=True)
        df['lag_56_price_norm']  = grp_p['price_norm'].shift(56).astype('float32')
        df['lag_56_is_discount'] = (df['lag_56_price_norm'] < 0.95).astype('int8')
        df['lag_56_margin_rate'] = grp_p['margin_rate'].shift(56).astype('float32')
        df.drop(columns=['implied_price', 'implied_cost', 'price_norm', 'margin_rate'], inplace=True)

    return df

print('Building v5 features (may take 2-3 min) ...')
daily = build_features(daily, VN_HOLIDAYS, TET_DAY1_LIST)
print(f'Total columns: {len(daily.columns)}')
print('Columns:', daily.columns.tolist())


---
##  Cell 8 — WRMSSE Metric

In [ ]:
def compute_wrmsse(y_true, y_pred, train_series, weights):
    """
    WRMSSE = sum(w_i * RMSSE_i)
    RMSSE_i = sqrt(MSE_forecast / MSE_naive)
    """
    eps          = 1e-9
    naive_errors = np.diff(train_series, axis=1) ** 2
    denom        = naive_errors.mean(axis=1)
    mse_forecast = ((y_true - y_pred) ** 2).mean(axis=1)
    rmsse        = np.sqrt(mse_forecast / (denom + eps))
    return float(np.sum(weights * rmsse))

print('WRMSSE defined')

---
## ️ Cell 9 — Split, Cross Features & Sample Weights

In [ ]:
EXCLUDE_COLS = ['Date', 'Quantity', 'SalesAmount', 'CostAmount']
FEATURE_COLS = [c for c in daily.columns if c not in EXCLUDE_COLS]
print(f'Base features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

train_df = daily[daily['Date'] <  SPLIT_DATE].copy()
val_df   = daily[(daily['Date'] >= SPLIT_DATE) & (daily['Date'] <= TRAIN_END)].copy()

# ── Expanded Cross Features (v5: week + quarter added) ────────────────────────
# All computed ONLY from training data (leak-free)
print('Computing v5 expanded cross features ...')
sku_map         = train_df.groupby('ItemCode', observed=True)['Quantity'].mean().to_dict()
dow_map         = train_df.groupby(['ItemCode', 'dayofweek'], observed=True)['Quantity'].mean().to_dict()
month_map       = train_df.groupby(['ItemCode', 'month'],     observed=True)['Quantity'].mean().to_dict()
week_map        = train_df.groupby(['ItemCode', 'weekofyear'],observed=True)['Quantity'].mean().to_dict()
quarter_map     = train_df.groupby(['ItemCode', 'quarter'],   observed=True)['Quantity'].mean().to_dict()

# EWM-smoothed DoW mean (captures recent day-of-week pattern shift)
# For each SKU×DoW, compute EWM over recent weeks
def make_sku_dow_ewm_map(df, span=12):
    """
    For each (ItemCode, dayofweek) pair, compute the EWM-weighted mean
    of the last `span` occurrences of that day-of-week for each SKU.
    Returns a dict (ItemCode, dow) -> ewm_mean
    """
    result = {}
    for (sku, dow), grp in df.groupby(['ItemCode', 'dayofweek'], observed=True):
        vals = grp['Quantity'].values
        if len(vals) == 0:
            result[(sku, dow)] = 0.0
            continue
        # Compute EWM mean of last `span` values
        alpha = 2.0 / (span + 1)
        ewm_val = float(vals[0])
        for v in vals[1:]:
            ewm_val = alpha * v + (1 - alpha) * ewm_val
        result[(sku, dow)] = ewm_val
    return result

print('  Building sku_dow_ewm map (may take 30-60s) ...')
dow_ewm_map = make_sku_dow_ewm_map(train_df, span=12)

def map_cross_features(df):
    df = df.copy()
    df['sku_mean_qty']   = df['ItemCode'].map(sku_map).fillna(0).astype('float32')

    dk = pd.MultiIndex.from_arrays([df['ItemCode'], df['dayofweek']])
    df['sku_dow_mean']   = dk.map(dow_map).fillna(0).astype('float32')
    df['sku_dow_ewm']    = dk.map(dow_ewm_map).fillna(0).astype('float32')

    mk = pd.MultiIndex.from_arrays([df['ItemCode'], df['month']])
    df['sku_month_mean'] = mk.map(month_map).fillna(0).astype('float32')

    wk = pd.MultiIndex.from_arrays([df['ItemCode'], df['weekofyear']])
    df['sku_week_mean']  = wk.map(week_map).fillna(0).astype('float32')

    qk = pd.MultiIndex.from_arrays([df['ItemCode'], df['quarter']])
    df['sku_quarter_mean'] = qk.map(quarter_map).fillna(0).astype('float32')

    return df

print('  Applying cross features ...')
daily    = map_cross_features(daily)
train_df = map_cross_features(train_df)
val_df   = map_cross_features(val_df)

new_cross = ['sku_mean_qty', 'sku_dow_mean', 'sku_dow_ewm',
             'sku_month_mean', 'sku_week_mean', 'sku_quarter_mean']
for f in new_cross:
    if f not in FEATURE_COLS:
        FEATURE_COLS.append(f)

print(f'Total features with cross: {len(FEATURE_COLS)}')

# ── Validation-window Profit Weights ─────────────────────────────────────────
train_raw = raw[raw['Date'] < SPLIT_DATE].copy()
train_raw['Profit'] = train_raw['SalesAmount'].astype(float) - train_raw['Cost Amount'].astype(float)
sku_profit_val = (
    train_raw.groupby('ItemCode', observed=True)['Profit']
       .sum().reset_index().rename(columns={'Profit': 'total_profit'})
)
sku_profit_val['total_profit'] = sku_profit_val['total_profit'].clip(lower=0)
total_p = sku_profit_val['total_profit'].sum()
sku_profit_val['weight'] = sku_profit_val['total_profit'] / total_p if total_p > 0 else 0.0

# ── Sample Weights: profit^0.7 / sqrt(denom), capped ─────────────────────────
train_pivot_denom = train_df.pivot(index='ItemCode', columns='Date', values='Quantity')
naive_err_w       = np.diff(train_pivot_denom.values, axis=1) ** 2
denom_vals_w      = naive_err_w.mean(axis=1)
denom_df_w        = pd.DataFrame({'ItemCode': train_pivot_denom.index, 'denom': denom_vals_w})

sku_stats_val = sku_profit_val.merge(denom_df_w, on='ItemCode', how='left')
for col in ['weight', 'denom', 'total_profit']:
    sku_stats_val[col] = sku_stats_val[col].fillna(0.0)

eps_w = 0.01
sku_stats_val['sample_weight'] = (
    sku_stats_val['total_profit'] ** 0.7 / np.sqrt(sku_stats_val['denom'] + eps_w)
)
pos_mask  = sku_stats_val['sample_weight'] > 0
median_sw = sku_stats_val.loc[pos_mask, 'sample_weight'].median()
sku_stats_val['sample_weight'] = sku_stats_val['sample_weight'].clip(upper=50 * median_sw)

weight_map    = sku_stats_val.set_index('ItemCode')['sample_weight'].to_dict()
train_weights = train_df['ItemCode'].map(weight_map).fillna(0.0).values
if train_weights.sum() > 0:
    train_weights = train_weights / train_weights.mean()

# Target: square root transform
X_train, y_train = train_df[FEATURE_COLS], np.sqrt(train_df['Quantity'])
X_val,   y_val   = val_df[FEATURE_COLS],   np.sqrt(val_df['Quantity'])

print(f'Train: {len(train_df):,} | {train_df["Date"].min().date()} -> {train_df["Date"].max().date()}')
print(f'Val  : {len(val_df):,} | {val_df["Date"].min().date()} -> {val_df["Date"].max().date()}')


---
##  Cell 10 — Model 1: SQRT Regression LightGBM

In [ ]:
lgb_train = lgb.Dataset(
    X_train, label=y_train, weight=train_weights,
    categorical_feature=['ItemCode'], free_raw_data=False
)
lgb_val = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=['ItemCode'], reference=lgb_train, free_raw_data=False
)

params_sqrt = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.05,
    'num_leaves':        255,
    'max_depth':         -1,
    'min_child_samples': 30,
    'feature_fraction':  0.75,
    'bagging_fraction':  0.85,
    'bagging_freq':      1,
    'lambda_l1':         0.05,
    'lambda_l2':         0.5,
    'path_smooth':       0.5,
    'verbose':           -1,
    'seed':              SEED,
    'n_jobs':            -1,
}
callbacks = [lgb.early_stopping(100, verbose=True), lgb.log_evaluation(100)]

print('Training Model 1 (SQRT Regression) ...')
model_sqrt = lgb.train(
    params_sqrt, lgb_train, num_boost_round=5000,
    valid_sets=[lgb_train, lgb_val], valid_names=['train', 'val'],
    callbacks=callbacks,
)
print(f'Model 1 best iteration: {model_sqrt.best_iteration}')


---
##  Cell 11 — Model 2: Tweedie (for sparse zero-inflated counts)

In [ ]:
# Tweedie distribution is designed for zero-inflated count data
# variance_power=1.5: between Poisson (p=1) and Gamma (p=2), good for demand
lgb_train_tw = lgb.Dataset(
    X_train, label=train_df['Quantity'].values,  # raw quantity, NOT sqrt
    weight=train_weights,
    categorical_feature=['ItemCode'], free_raw_data=False
)
lgb_val_tw = lgb.Dataset(
    X_val, label=val_df['Quantity'].values,       # raw quantity
    categorical_feature=['ItemCode'], reference=lgb_train_tw, free_raw_data=False
)

params_tweedie = {
    'objective':              'tweedie',
    'tweedie_variance_power': 1.5,
    'metric':                 'tweedie',
    'learning_rate':          0.05,
    'num_leaves':             255,
    'max_depth':              -1,
    'min_child_samples':      30,
    'feature_fraction':       0.75,
    'bagging_fraction':       0.85,
    'bagging_freq':           1,
    'lambda_l1':              0.05,
    'lambda_l2':              0.5,
    'path_smooth':            0.5,
    'verbose':                -1,
    'seed':                   SEED + 1,
    'n_jobs':                 -1,
}
callbacks_tw = [lgb.early_stopping(100, verbose=True), lgb.log_evaluation(100)]

print('Training Model 2 (Tweedie) ...')
model_tweedie = lgb.train(
    params_tweedie, lgb_train_tw, num_boost_round=5000,
    valid_sets=[lgb_train_tw, lgb_val_tw], valid_names=['train', 'val'],
    callbacks=callbacks_tw,
)
print(f'Model 2 best iteration: {model_tweedie.best_iteration}')


---
##  Cell 12 — Validation, Ensemble & Data-Driven Calibration

In [ ]:
# ── Model 1: SQRT predictions ────────────────────────────────────────────────
val_sqrt_preds  = np.square(np.clip(
    model_sqrt.predict(X_val, num_iteration=model_sqrt.best_iteration), 0, None
))

# ── Model 2: Tweedie predictions ──────────────────────────────────────────────
val_tweedie_preds = np.clip(
    model_tweedie.predict(X_val, num_iteration=model_tweedie.best_iteration), 0, None
)

# ── Ensemble (65% SQRT + 35% Tweedie) ────────────────────────────────────────
BLEND_SQRT    = 0.65
BLEND_TWEEDIE = 0.35
val_blend = BLEND_SQRT * val_sqrt_preds + BLEND_TWEEDIE * val_tweedie_preds

val_df = val_df.copy()
val_df['pred_sqrt']    = val_sqrt_preds
val_df['pred_tweedie'] = val_tweedie_preds
val_df['pred']         = val_blend

# ── Sparse SKU zeroing ────────────────────────────────────────────────────────
active_days_map = daily.groupby('ItemCode')['sku_active_days'].first().to_dict()
val_df['active_days'] = val_df['ItemCode'].map(active_days_map).fillna(0)
train_sums_v    = train_df.groupby('ItemCode')['Quantity'].sum().to_dict()
val_df['train_sum'] = val_df['ItemCode'].map(train_sums_v).fillna(0.0)
val_df.loc[val_df['train_sum'] == 0, 'pred'] = 0.0
val_df.loc[val_df['active_days'] < 5, 'pred'] = 0.0

# ── Data-driven calibration per profit tier ───────────────────────────────────
# Instead of arbitrary multipliers (1.20, 1.10), compute the ratio of
# actual vs predicted within each tier on the validation set.
profit_wmap_v = sku_profit_val.set_index('ItemCode')['weight'].to_dict()
val_df['profit_weight'] = val_df['ItemCode'].map(profit_wmap_v).fillna(0.0)
pos_pw = val_df['profit_weight'][val_df['profit_weight'] > 0]
w95_v  = pos_pw.quantile(0.95)
w80_v  = pos_pw.quantile(0.80)
w50_v  = pos_pw.quantile(0.50)

val_df['tier'] = 0
val_df.loc[val_df['profit_weight'] >= w95_v, 'tier'] = 3
val_df.loc[(val_df['profit_weight'] >= w80_v) & (val_df['profit_weight'] < w95_v), 'tier'] = 2
val_df.loc[(val_df['profit_weight'] >= w50_v) & (val_df['profit_weight'] < w80_v), 'tier'] = 1

calib_factors = {}
for tier in [1, 2, 3]:
    mask = (val_df['tier'] == tier) & (val_df['pred'] > 0)
    if mask.sum() > 0:
        actual_mean = val_df.loc[mask, 'Quantity'].mean()
        pred_mean   = val_df.loc[mask, 'pred'].mean()
        factor      = actual_mean / (pred_mean + 1e-9)
        # Clip to [0.8, 1.5] to avoid extreme corrections
        calib_factors[tier] = float(np.clip(factor, 0.8, 1.5))
    else:
        calib_factors[tier] = 1.0

print('Data-driven calibration factors (actual/predicted per tier):')
for tier, factor in calib_factors.items():
    mask = val_df['tier'] == tier
    print(f'  Tier {tier}: factor={factor:.4f} ({mask.sum():,} rows)')

# Apply calibration
for tier, factor in calib_factors.items():
    mask = val_df['tier'] == tier
    val_df.loc[mask, 'pred'] *= factor

# ── WRMSSE Computation ────────────────────────────────────────────────────────
y_true_matrix = val_df.pivot(index='ItemCode', columns='Date', values='Quantity').values
y_pred_matrix = val_df.pivot(index='ItemCode', columns='Date', values='pred').values
val_sku_order = val_df.pivot(index='ItemCode', columns='Date', values='Quantity').index
train_pivot   = (
    train_df.pivot(index='ItemCode', columns='Date', values='Quantity')
    .reindex(val_sku_order).fillna(0).values
)
wm_val   = sku_profit_val.set_index('ItemCode')['weight'].to_dict()
val_wts  = np.array([wm_val.get(s, 0.0) for s in val_sku_order])
if val_wts.sum() > 0:
    val_wts /= val_wts.sum()

wrmsse_score = compute_wrmsse(y_true_matrix, y_pred_matrix, train_pivot, val_wts)

# Also score individual models for comparison
val_df['pred_sqrt_cal']    = val_sqrt_preds
val_df.loc[val_df['train_sum'] == 0, 'pred_sqrt_cal'] = 0.0
val_df.loc[val_df['active_days'] < 5, 'pred_sqrt_cal'] = 0.0
y_sq_mat  = val_df.pivot(index='ItemCode', columns='Date', values='pred_sqrt_cal').reindex(val_sku_order).fillna(0).values
wrmsse_sqrt = compute_wrmsse(y_true_matrix, y_sq_mat, train_pivot, val_wts)

val_df['pred_tw_cal']    = val_tweedie_preds
val_df.loc[val_df['train_sum'] == 0, 'pred_tw_cal'] = 0.0
val_df.loc[val_df['active_days'] < 5, 'pred_tw_cal'] = 0.0
y_tw_mat  = val_df.pivot(index='ItemCode', columns='Date', values='pred_tw_cal').reindex(val_sku_order).fillna(0).values
wrmsse_tweedie = compute_wrmsse(y_true_matrix, y_tw_mat, train_pivot, val_wts)

print(f'\n--- Validation WRMSSE Results ---')
print(f'  SQRT model only  : {wrmsse_sqrt:.4f}')
print(f'  Tweedie model    : {wrmsse_tweedie:.4f}')
print(f'  Ensemble (blend) : {wrmsse_score:.4f}')
print(f'  v3 baseline      : 0.5585')
print(f'  Delta vs v3      : {wrmsse_score - 0.5585:+.4f}')


---
##  Cell 13 — Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance_df = pd.DataFrame({
    'feature':    model_sqrt.feature_name(),
    'importance': model_sqrt.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)
print('Top 30 features by gain:')
print(importance_df.head(30).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 12))
top30 = importance_df.head(30)
ax.barh(top30['feature'][::-1], top30['importance'][::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title('LightGBM v5 Feature Importance — Top 30')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance_v5.png', dpi=120)
plt.show()

---
##  Cell 14 — Forecast Setup

In [ ]:
all_skus  = daily['ItemCode'].cat.categories.tolist()
all_forecast_dates = pd.date_range(VAL_START, EVAL_END, freq='D')
print(f'Forecast dates: {all_forecast_dates[0].date()} to {all_forecast_dates[-1].date()}')
print(f'Total inference: {len(all_skus):,} x {len(all_forecast_dates)} = {len(all_skus)*len(all_forecast_dates):,} rows')

---
##  Cell 15 — 56-Day Ensemble Inference

In [ ]:
print('Generating v5 optimized 56-day forecasts ...')

# Pre-compute full history pivot ONCE
HISTORY_START = TRAIN_END - pd.Timedelta(days=400)
history_df    = daily[daily['Date'] >= HISTORY_START][[
    'ItemCode', 'Date', 'Quantity', 'lag_56_price_norm',
    'lag_56_is_discount', 'lag_56_margin_rate', 'lag_56_days_since_sale'
]].copy()
hist_pivot   = history_df.pivot(index='ItemCode', columns='Date', values='Quantity').fillna(0)
hist_cols_set= set(hist_pivot.columns)

holiday_dates_sorted = sorted(VN_HOLIDAYS)
train_sums           = daily.groupby('ItemCode')['Quantity'].sum().to_dict()
active_days_map_inf  = daily.groupby('ItemCode')['sku_active_days'].first().to_dict()
freq_map             = daily.set_index('ItemCode')['sku_sales_freq'].to_dict()

# Cross-feature maps (from training data only)
train_mask      = daily['Date'] < SPLIT_DATE
sku_map_inf     = daily[train_mask].groupby('ItemCode', observed=True)['Quantity'].mean().to_dict()

# Static SKU stats
# sku_recent_mean_90d for inference: use last 90d of training data
recent_90d_start = TRAIN_END - pd.Timedelta(days=90)
recent_90d_mask  = (daily['Date'] >= recent_90d_start) & (daily['Date'] <= TRAIN_END)
sku_recent_map   = daily[recent_90d_mask].groupby('ItemCode', observed=True)['Quantity'].mean().to_dict()
sku_mean_all_map = daily.groupby('ItemCode', observed=True)['Quantity'].mean().to_dict()
sku_growth_map   = {
    sku: sku_recent_map.get(sku, 0.0) / (sku_mean_all_map.get(sku, 0.0) + 1e-6)
    for sku in sku_mean_all_map
}

# Price/margin/streak from last historical value
last_price_norm_map  = history_df.groupby('ItemCode')['lag_56_price_norm'].last().to_dict()
last_is_discount_map = history_df.groupby('ItemCode')['lag_56_is_discount'].last().to_dict()
last_margin_rate_map = history_df.groupby('ItemCode')['lag_56_margin_rate'].last().to_dict()
last_streak_map      = history_df.groupby('ItemCode')['lag_56_days_since_sale'].last().to_dict()

# Calibration tiers (profit weights)
profit_wmap_inf = sku_profit_val.set_index('ItemCode')['weight'].to_dict()
all_weights_arr = np.array([profit_wmap_inf.get(s, 0.0) for s in all_skus])
pos_w = all_weights_arr[all_weights_arr > 0]
w95_inf = float(np.quantile(pos_w, 0.95)) if len(pos_w) else 1.0
w80_inf = float(np.quantile(pos_w, 0.80)) if len(pos_w) else 1.0
w50_inf = float(np.quantile(pos_w, 0.50)) if len(pos_w) else 1.0

m_t3_inf = all_weights_arr >= w95_inf
m_t2_inf = (all_weights_arr >= w80_inf) & ~m_t3_inf
m_t1_inf = (all_weights_arr >= w50_inf) & ~m_t3_inf & ~m_t2_inf

# Days since TRAIN_START for trend_t feature
TRAIN_START_TS_REF = TRAIN_START

all_forecast_records = []

for i, fdate in enumerate(all_forecast_dates):
    if (i + 1) % 10 == 0 or i == 0:
        print(f'  Day {i+1:2d}/56: {fdate.date()}')

    future_hols         = [h for h in holiday_dates_sorted if h > fdate]
    past_hols           = [h for h in holiday_dates_sorted if h < fdate]
    days_to_next_hol    = (future_hols[0] - fdate).days if future_hols else 365
    days_since_last_hol = (fdate - past_hols[-1]).days  if past_hols  else 365

    future_tet = [t for t in TET_DAY1_LIST if t > fdate]
    past_tet   = [t for t in TET_DAY1_LIST if t <= fdate]
    d_to_tet   = (future_tet[0] - fdate).days if future_tet else 365
    d_from_tet = (fdate - past_tet[-1]).days   if past_tet   else 365

    row_df = pd.DataFrame({'ItemCode': all_skus})

    # Lag features (pruned: 56, 63, 91, 364 only)
    for lag in [56, 63, 91, 364]:
        lag_date = fdate - pd.Timedelta(days=lag)
        if lag_date in hist_cols_set:
            row_df[f'lag_{lag}'] = hist_pivot[lag_date].reindex(all_skus).fillna(0).values
        else:
            row_df[f'lag_{lag}'] = 0.0

    # EWM features from lag-56 window (expanded spans)
    lag56_date = fdate - pd.Timedelta(days=56)
    for span in [7, 14, 28, 56, 91]:
        span_start = lag56_date - pd.Timedelta(days=span * 4)
        dates_ewm  = [d for d in hist_pivot.columns if span_start <= d <= lag56_date]
        if dates_ewm:
            ev = hist_pivot[dates_ewm].reindex(all_skus).fillna(0)
            row_df[f'ewm_mean_{span}'] = ev.apply(
                lambda x, s=span: x.ewm(span=s, min_periods=1).mean().iloc[-1], axis=1
            ).values
        else:
            row_df[f'ewm_mean_{span}'] = 0.0

    # Ratio features
    eps_r = 1e-3
    row_df['ewm_ratio_14_56'] = (row_df['ewm_mean_14'] / (row_df['ewm_mean_56']  + eps_r)).clip(0, 10)
    row_df['ewm_ratio_28_91'] = (row_df['ewm_mean_28'] / (row_df['ewm_mean_91']  + eps_r)).clip(0, 10)
    row_df['lag56_vs_ewm56']  = (row_df['lag_56']      / (row_df['ewm_mean_56']  + eps_r)).clip(0, 10)

    # Rolling features from lag-56 window
    for window in [28, 56]:
        w_start   = lag56_date - pd.Timedelta(days=window - 1)
        diw = [d for d in hist_pivot.columns if w_start <= d <= lag56_date]
        if diw:
            wv = hist_pivot[diw].reindex(all_skus).fillna(0)
            row_df[f'roll_mean_{window}'] = wv.mean(axis=1).values
        else:
            row_df[f'roll_mean_{window}'] = 0.0

    # roll_max_56
    max56_start = lag56_date - pd.Timedelta(days=55)
    diw_max = [d for d in hist_pivot.columns if max56_start <= d <= lag56_date]
    if diw_max:
        wv_max = hist_pivot[diw_max].reindex(all_skus).fillna(0)
        row_df['roll_max_56'] = wv_max.max(axis=1).values
    else:
        row_df['roll_max_56'] = 0.0

    # roll_nonzero_28
    nz28_start = lag56_date - pd.Timedelta(days=27)
    diw_nz = [d for d in hist_pivot.columns if nz28_start <= d <= lag56_date]
    if diw_nz:
        wv_nz = hist_pivot[diw_nz].reindex(all_skus).fillna(0)
        row_df['roll_nonzero_28'] = (wv_nz > 0).sum(axis=1).values
    else:
        row_df['roll_nonzero_28'] = 0.0

    # roll_std_56
    std56_start = lag56_date - pd.Timedelta(days=55)
    diw_std = [d for d in hist_pivot.columns if std56_start <= d <= lag56_date]
    if diw_std:
        wv_std = hist_pivot[diw_std].reindex(all_skus).fillna(0)
        row_df['roll_std_56'] = wv_std.std(axis=1).fillna(0).values
    else:
        row_df['roll_std_56'] = 0.0

    # Static SKU features
    row_df['sku_sales_freq']  = row_df['ItemCode'].map(freq_map).fillna(0).values
    row_df['sku_active_days'] = row_df['ItemCode'].map(active_days_map_inf).fillna(0).values
    row_df['sku_recent_mean_90d'] = row_df['ItemCode'].map(sku_recent_map).fillna(0).values
    row_df['sku_growth_rate'] = row_df['ItemCode'].map(sku_growth_map).fillna(1.0).clip(0, 5).values

    # Calendar features
    iso_w = fdate.isocalendar()[1]
    trend_t_val = (fdate - TRAIN_START_TS_REF).days
    time_feats = {
        'dayofweek':               fdate.dayofweek,
        'day':                     fdate.day,
        'month':                   fdate.month,
        'year':                    fdate.year,
        'quarter':                 fdate.quarter,
        'weekofyear':              iso_w,
        'is_weekend':              int(fdate.dayofweek >= 5),
        'is_holiday':              int(fdate in VN_HOLIDAYS),
        'days_to_next_holiday':    days_to_next_hol,
        'days_since_last_holiday': days_since_last_hol,
        'is_pre_holiday_7d':       1 if (0 < days_to_next_hol  <= 7) else 0,
        'is_post_holiday_3d':      1 if (0 < days_since_last_hol <= 3) else 0,
        'days_to_tet':             min(d_to_tet, 365),
        'days_since_tet':          min(d_from_tet, 365),
        'trend_t':                 trend_t_val,
    }
    for k, v in time_feats.items():
        row_df[k] = v

    # Cross features
    row_df['sku_mean_qty'] = row_df['ItemCode'].map(sku_map_inf).fillna(0).astype('float32')
    dk = pd.MultiIndex.from_arrays([row_df['ItemCode'], row_df['dayofweek']])
    row_df['sku_dow_mean']  = dk.map(dow_map).fillna(0).astype('float32')
    row_df['sku_dow_ewm']   = dk.map(dow_ewm_map).fillna(0).astype('float32')
    mk = pd.MultiIndex.from_arrays([row_df['ItemCode'], row_df['month']])
    row_df['sku_month_mean']= mk.map(month_map).fillna(0).astype('float32')
    wk = pd.MultiIndex.from_arrays([row_df['ItemCode'], row_df['weekofyear']])
    row_df['sku_week_mean'] = wk.map(week_map).fillna(0).astype('float32')
    qk = pd.MultiIndex.from_arrays([row_df['ItemCode'], row_df['quarter']])
    row_df['sku_quarter_mean'] = qk.map(quarter_map).fillna(0).astype('float32')

    # Price/margin/streak
    row_df['lag_56_price_norm']      = row_df['ItemCode'].map(last_price_norm_map).fillna(1.0).values
    row_df['lag_56_is_discount']     = row_df['ItemCode'].map(last_is_discount_map).fillna(0).values
    row_df['lag_56_margin_rate']     = row_df['ItemCode'].map(last_margin_rate_map).fillna(0).values
    row_df['lag_56_days_since_sale'] = row_df['ItemCode'].map(last_streak_map).fillna(0).values + i

    row_df['Date']     = fdate
    row_df['ItemCode'] = row_df['ItemCode'].astype('category')

    # SQRT model prediction
    preds_sqrt = np.square(np.clip(
        model_sqrt.predict(row_df[FEATURE_COLS], num_iteration=model_sqrt.best_iteration), 0, None
    ))
    # Tweedie model prediction
    preds_tw = np.clip(
        model_tweedie.predict(row_df[FEATURE_COLS], num_iteration=model_tweedie.best_iteration), 0, None
    )
    # Ensemble blend
    preds = BLEND_SQRT * preds_sqrt + BLEND_TWEEDIE * preds_tw
    row_df['forecast'] = preds

    # Sparse SKU zeroing
    row_df['train_sum']    = row_df['ItemCode'].map(train_sums).fillna(0.0)
    row_df['active_d']     = row_df['ItemCode'].map(active_days_map_inf).fillna(0)
    row_df.loc[row_df['train_sum'] == 0, 'forecast'] = 0.0
    row_df.loc[row_df['active_d'] < 5, 'forecast'] = 0.0

    # Apply calibration factors (data-driven from validation)
    row_df.loc[m_t3_inf, 'forecast'] *= calib_factors.get(3, 1.0)
    row_df.loc[m_t2_inf, 'forecast'] *= calib_factors.get(2, 1.0)
    row_df.loc[m_t1_inf, 'forecast'] *= calib_factors.get(1, 1.0)

    all_forecast_records.append(row_df[['ItemCode', 'Date', 'forecast']])

forecast_df = pd.concat(all_forecast_records, ignore_index=True)
print(f'Forecast complete: {len(forecast_df):,} rows')


---
##  Cell 16 — Build & Validate Submission

In [ ]:
sample_sub = pd.read_csv(SAMPLE_PATH)
assert sample_sub.shape[1] == 29

F_COLS     = [f'F{i}' for i in range(1, 29)]
val_dates  = pd.date_range(VAL_START,  VAL_END,  freq='D')
eval_dates = pd.date_range(EVAL_START, EVAL_END, freq='D')
assert len(val_dates) == 28 and len(eval_dates) == 28

fc_pivot = forecast_df.pivot(index='ItemCode', columns='Date', values='forecast').fillna(0)

submission_rows = []
for idx_row in sample_sub.itertuples(index=False):
    row_id = idx_row.id
    if '_validation' in row_id:
        sku, dates = row_id.replace('_validation', ''), val_dates
    elif '_evaluation' in row_id:
        sku, dates = row_id.replace('_evaluation', ''), eval_dates
    else:
        raise ValueError(f'Unexpected id: {row_id}')

    preds_28 = np.clip(
        fc_pivot.loc[sku, dates].values if sku in fc_pivot.index else np.zeros(28), 0, None
    )
    row = {'id': row_id}
    row.update({f'F{j+1}': float(preds_28[j]) for j in range(28)})
    submission_rows.append(row)

submission = pd.DataFrame(submission_rows)
assert len(submission) == len(sample_sub)
assert set(submission['id']) == set(sample_sub['id'])
assert submission[F_COLS].isna().sum().sum() == 0
assert (submission[F_COLS] < 0).sum().sum() == 0
assert submission['id'].duplicated().sum() == 0
print('All checks passed!')

out_path = OUTPUT_DIR / 'submission_v5.csv'
submission.to_csv(out_path, index=False)
flat = submission[F_COLS].values.flatten()
print(f'Saved: {out_path}  ({out_path.stat().st_size/1024:.1f} KB)')
print(f'Stats: mean={flat.mean():.3f}, median={np.median(flat):.3f}, % zero={(flat==0).mean()*100:.1f}%')


---
##  Cell 17 — Summary

In [ ]:
print('=' * 60)
print('  v3 baseline WRMSSE: 0.5585')
print(f'  v5 SQRT only      : {wrmsse_sqrt:.4f}')
print(f'  v5 Tweedie only   : {wrmsse_tweedie:.4f}')
print(f'  v5 Ensemble final : {wrmsse_score:.4f}')
print(f'  Improvement vs v3 : {0.5585 - wrmsse_score:+.4f}')
print('=' * 60)
print(f'  Feature count: {len(FEATURE_COLS)}')
print(f'  SQRT model best iter : {model_sqrt.best_iteration}')
print(f'  Tweedie model best it: {model_tweedie.best_iteration}')
print(f'  Calib factors: {calib_factors}')
print()
for p in sorted(OUTPUT_DIR.glob('*.csv')) + sorted(OUTPUT_DIR.glob('*.png')):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')